# DDRI cluster01 3차 진행표 생성 노트북

- 목적: `cluster01`의 1차, 2차, 3차 최종 test(최종 평가)(최종 평가) 성능을 한 표로 모아 `cluster01_third_round_progression_summary.csv`를 재생성한다.
- 입력:
  - `summary_aggregation/output/data/ddri_cluster_model_metrics_collection_template.csv`
  - `summary_aggregation/output/data/ddri_cluster_second_round_comparison_summary.csv`
  - `works/05_prediction_long/output/team_cluster_third_round_outputs/ddri_아침_도착_업무_집중형_third_round_model_metrics.csv`
- 출력: `summary_aggregation/output/data/cluster01_third_round_progression_summary.csv`


## 용어 설명
- `summary(요약 정본)`: 군집별 실험 결과를 한 번에 모아 비교하기 위한 집계 문서
- `test(최종 평가)`: 최종 재학습 뒤 2025 데이터로 점검한 결과
- `progression(진행 경과)`: 1차, 2차, 3차처럼 실험이 어떻게 변해 왔는지 연결해 보는 정리 방식
- `subset(축소 피처 조합)`: 전체 피처 중 일부만 남겨 만든 비교 조합


In [1]:

from pathlib import Path
import pandas as pd

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
AGG_DIR = ROOT / 'works/05_prediction_long/cheng80/summary_aggregation'
OUTPUT_DATA_DIR = AGG_DIR / 'output/data'
FIRST_PATH = OUTPUT_DATA_DIR / 'ddri_cluster_model_metrics_collection_template.csv'
SECOND_PATH = OUTPUT_DATA_DIR / 'ddri_cluster_second_round_comparison_summary.csv'
THIRD_PATH = ROOT / 'works/05_prediction_long/output/team_cluster_third_round_outputs/ddri_아침_도착_업무_집중형_third_round_model_metrics.csv'

first_df = pd.read_csv(FIRST_PATH, encoding='utf-8-sig')
second_df = pd.read_csv(SECOND_PATH, encoding='utf-8-sig')
third_df = pd.read_csv(THIRD_PATH, encoding='utf-8-sig')


## 1. cluster01 1~3차 최종 test(최종 평가)(최종 평가) 결과 연결

In [2]:

first = first_df[(first_df['cluster_code'] == 'cluster01') & (first_df['best_model_flag'] == 1)].iloc[0]
second = second_df[second_df['cluster_code'] == 'cluster01'].iloc[0]
third = third_df[third_df['split'] == 'test_2025_refit'].sort_values('rmse').iloc[0]

progression = pd.DataFrame([
    {
        'round': 'first',
        'model': first['model'],
        'test_rmse': float(first['rmse']),
        'test_mae': float(first['mae']),
        'test_r2': float(first['r2']),
    },
    {
        'round': 'second',
        'model': second['second_round_best_model'],
        'test_rmse': float(second['second_round_test_rmse']),
        'test_mae': float(second['second_round_test_mae']),
        'test_r2': float(second['second_round_test_r2']),
    },
    {
        'round': 'third',
        'model': third['model'],
        'test_rmse': float(third['rmse']),
        'test_mae': float(third['mae']),
        'test_r2': float(third['r2']),
    },
])
first_rmse = progression.loc[0, 'test_rmse']
first_mae = progression.loc[0, 'test_mae']
first_r2 = progression.loc[0, 'test_r2']
progression['rmse_delta_vs_first'] = (progression['test_rmse'] - first_rmse).round(6)
progression['mae_delta_vs_first'] = (progression['test_mae'] - first_mae).round(6)
progression['r2_delta_vs_first'] = (progression['test_r2'] - first_r2).round(6)

out_path = OUTPUT_DATA_DIR / 'cluster01_third_round_progression_summary.csv'
progression.to_csv(out_path, index=False, encoding='utf-8-sig')
print(progression.to_string(index=False))
print() 
print('saved:', out_path)


 round            model  test_rmse  test_mae  test_r2  rmse_delta_vs_first  mae_delta_vs_first  r2_delta_vs_first
 first    LightGBM_RMSE   1.346200  0.804200  0.63980             0.000000            0.000000            0.00000
second    LightGBM_RMSE   1.332400  0.786800  0.64710            -0.013800           -0.017400            0.00730
 third LightGBM_Poisson   1.318903  0.774514  0.65427            -0.027297           -0.029686            0.01447

saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/summary_aggregation/output/data/cluster01_third_round_progression_summary.csv
